# Cleaning Schedule as a CSP

A constraint-satisfaction problem has variables, domains, and rules restricting valid combinations. Here, each dirty location is a task variable (`D1`, `D2`, ...), each time slot is a possible value, precedence rules require one task to come earlier, latest-slot rules set deadlines, and exclusivity prevents two tasks from using the same slot.

Tasks come from `Environment.txt`; slot counts and constraints come from `schedule.txt`. The solver uses MRV with a degree tie-break, forward checking, least-constraining-value ordering, and recursive backtracking.

In [16]:
import sys
from pathlib import Path

workspace_root = next(
    (parent for parent in (Path.cwd(), *Path.cwd().parents)
     if (parent / "core" / "environment.py").is_file()),
    None,
)
if workspace_root is None:
    raise RuntimeError("Could not find the workspace core/environment.py")
if str(workspace_root) not in sys.path:
    sys.path.insert(0, str(workspace_root))

from core.environment import load_environment

In [17]:
from typing import Optional

class ScheduleFormatError(Exception):
    """Raised when a schedule file is malformed."""

    def __init__(self, message: str, line_number: Optional[int] = None) -> None:
        self.line_number = line_number
        full = f"Line {line_number}: {message}" if line_number is not None else message
        super().__init__(full)

## Build task variables from the environment

The environment loader provides dirty-location coordinates in file order. We assign labels `D1`, `D2`, and so on from that order, so every dirty location is represented, including tasks absent from the constraints.

In [18]:
#generating variables from the environment

grid=load_environment(workspace_root / "notebook_cells" / "Environment.txt")
variables=[f"D{i+1}" for i in range(len(grid.dirty_locations))]
print(grid)
print(variables)

Grid(rows=10, cols=10, obstacles={(1, 2), (5, 5), (6, 5), (4, 2), (4, 5), (2, 2), (3, 2), (2, 5), (3, 5), (5, 2)}, dirty_locations=[(0, 9), (4, 9), (9, 9), (9, 0), (5, 7)], charging_station=(0, 0))
['D1', 'D2', 'D3', 'D4', 'D5']


## Read the scheduling problem

`load_schedule` reads the slot count, precedence count and pairs, then latest-slot count and pairs. It creates a separate copy of the full slot list for each task. Reading exactly the declared number of pairs supports different constraint counts and avoids depending on a fixed task or slot total.

In [19]:
import re
from pathlib import Path
from typing import Union
def load_schedule(variables,filepath: Union[str, Path]):
    """Load slots and constraints from the count-based schedule file."""
    text = Path(filepath).read_text(encoding="utf-8")
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    if len(lines) < 3:
        raise ScheduleFormatError("File too short")

    try:
        n = int(lines[0])
        precedence_count = int(lines[1])
    except ValueError as exc:
        raise ScheduleFormatError("Slot and precedence counts must be integers") from exc
    if n < 1 or precedence_count < 0:
        raise ScheduleFormatError("Slot count must be positive and constraint counts non-negative")

    idx = 2
    precedences = []
    for _ in range(precedence_count):
        if idx >= len(lines):
            raise ScheduleFormatError("File ended before all precedence constraints were read")
        match = re.fullmatch(r"\(\s*(D\d+)\s*,\s*(D\d+)\s*\)", lines[idx])
        if match is None:
            raise ScheduleFormatError(f"Invalid precedence constraint: {lines[idx]!r}")
        precedences.append((match.group(1), match.group(2)))
        idx += 1

    if idx >= len(lines):
        raise ScheduleFormatError("Missing latest-slot constraint count")
    try:
        latest_count = int(lines[idx])
    except ValueError as exc:
        raise ScheduleFormatError("Latest-slot constraint count must be an integer") from exc
    if latest_count < 0:
        raise ScheduleFormatError("Latest-slot constraint count cannot be negative")
    idx += 1

    latest_con = {}
    for _ in range(latest_count):
        if idx >= len(lines):
            raise ScheduleFormatError("File ended before all latest-slot constraints were read")
        match = re.fullmatch(r"\(\s*(D\d+)\s*,\s*T(\d+)\s*\)", lines[idx])
        if match is None:
            raise ScheduleFormatError(f"Invalid latest-slot constraint: {lines[idx]!r}")
        latest_con[match.group(1)] = int(match.group(2))
        idx += 1
    if idx != len(lines):
        raise ScheduleFormatError(f"Unexpected extra schedule input: {lines[idx]!r}")

    slots = list(range(1, n + 1))
    domains = {v: list(slots) for v in variables}

    return domains, precedences, latest_con

The next cell displays the parsed CSP. Each precedence pair is ordered as `(task_that_must_be_earlier, task_that_must_be_later)`. The constraints dictionary uses the key `precedences`, which is the key used by the solver functions.

In [20]:
domains,precedences,latest=load_schedule(variables, workspace_root / "notebook_cells" / "schedule.txt")
print(f"domains: {domains}")
print(f"Precedence: {precedences}")
print(f"latest: {latest}")
constraints={}
constraints["precedences"]=precedences
constraints["latest"]=latest
print(constraints)

domains: {'D1': [1, 2, 3, 4, 5, 6, 7], 'D2': [1, 2, 3, 4, 5, 6, 7], 'D3': [1, 2, 3, 4, 5, 6, 7], 'D4': [1, 2, 3, 4, 5, 6, 7], 'D5': [1, 2, 3, 4, 5, 6, 7]}
Precedence: [('D1', 'D5'), ('D4', 'D1'), ('D3', 'D5'), ('D2', 'D4')]
latest: {'D1': 3, 'D5': 6, 'D3': 4}
{'precedences': [('D1', 'D5'), ('D4', 'D1'), ('D3', 'D5'), ('D2', 'D4')], 'latest': {'D1': 3, 'D5': 6, 'D3': 4}}


## Apply latest-slot limits

A limit such as `D1 by T3` removes slots after 3 from D1's domain, while keeping T3 itself. Tasks without a deadline keep their complete domain.

In [21]:
def apply_latest_slot(domains,latest):
    for task in domains.keys():
        if task in latest:
            
            domains[task]=[x for x in domains[task] if x<= latest[task] ]

    return domains

In [22]:
domains, precedences, latest = load_schedule(variables, workspace_root / "notebook_cells" / "schedule.txt")
apply_latest_slot(domains, latest)

{'D1': [1, 2, 3],
 'D2': [1, 2, 3, 4, 5, 6, 7],
 'D3': [1, 2, 3, 4],
 'D4': [1, 2, 3, 4, 5, 6, 7],
 'D5': [1, 2, 3, 4, 5, 6]}

## Check a candidate assignment

`is_consistent` rejects a value if it misses the task's deadline, reuses an occupied slot, or violates a precedence relation whose other task is already assigned. Precedence is strict: the earlier task's slot number must be smaller.

In [23]:
def is_consistent(variable, value, assignment, constraints):
    latest = constraints["latest"]
    precedences = constraints["precedences"]

    if variable in latest and value > latest[variable]:
        return False

    if value in assignment.values():
        return False

    for before, after in precedences:
        if variable == before and after in assignment:
            if value >= assignment[after]:
                return False
        elif variable == after and before in assignment:
            if value <= assignment[before]:
                return False

    return True

# constraints = {"latest": latest, "precedences": precedences}
# test_cases = [
#     ("D1", 5, {}, False),
#     ("D1", 3, {}, True),
#     ("D4", 3, {"D1": 3}, False),
#     ("D4", 4, {"D1": 3}, False),
#     ("D4", 2, {"D1": 3}, True),
#     ("D5", 2, {"D1": 3}, False),
#     ("D5", 5, {"D1": 3}, True),
# ]

# for variable, value, assignment, expected in test_cases:
#     result = is_consistent(variable, value, assignment, constraints)
#     print(f"{variable}={value}, assignment={assignment}: {result} (expected {expected})")
#     assert result is expected

## Forward checking

After assigning a task, `forward_check` copies the domains so the caller's domains remain unchanged. It removes the chosen slot from other unassigned tasks and narrows precedence neighbors to slots on the correct side. If any remaining task has no possible slot, it returns `None` so search can backtrack.

In [24]:
def forward_check(variable, value, assignment, domains, constraints):
    new_domains = {v: list(d) for v, d in domains.items()}
    precedences = constraints["precedences"]

    for other in new_domains:
        if other == variable or other in assignment:
            continue                      # only unassigned others

        # Rule 1: exclusivity -> 
        if value in new_domains[other]:

            new_domains[other].remove(value)


        # Rule 2: precedence, only for pairs linking variable and other
        for before, after in precedences:
            if before == variable and after == other:
                new_domains[other]=[x for x in new_domains[after] if x>value]

                
            elif after == variable and before == other:
                new_domains[other]=[x for x in new_domains[before] if x<value]
                

        if not new_domains[other]:        # empty domain -> reject
            return None

    return new_domains





### Forward-checking tests

These assertions check exclusivity, both directions of precedence pruning, detection of an empty domain, and that pruning does not mutate the original domains.

In [25]:
step0 = {
    "D1": [1, 2, 3],
    "D2": [1, 2, 3, 4, 5, 6, 7],
    "D3": [1, 2, 3, 4],
    "D4": [1, 2, 3, 4, 5, 6, 7],
    "D5": [1, 2, 3, 4, 5, 6],
}
constraints = {
    "latest": {"D1": 3, "D5": 6, "D3": 4},
    "precedences": [("D1","D5"), ("D4","D1"), ("D3","D5"), ("D2","D4")],
}

# Test 1: D1=3. Exclusivity hits D2/D3, precedence hits D4/D5
r = forward_check("D1", 3, {"D1": 3}, step0, constraints)
assert r["D2"] == [1, 2, 4, 5, 6, 7]
assert r["D3"] == [1, 2, 4]
assert r["D4"] == [1, 2]
assert r["D5"] == [4, 5, 6]
assert r["D1"] == [1, 2, 3]          # assigned variable is not touched

# Test 2: D4=7 leaves D1 nothing (D1 must come after D4) -> failure
assert forward_check("D4", 7, {"D4": 7}, step0, constraints) is None

# Test 3: restore -- the original dictionary must be unchanged
assert step0["D2"] == [1, 2, 3, 4, 5, 6, 7]
assert step0["D5"] == [1, 2, 3, 4, 5, 6]

# Test 4: value missing from some domains must not crash
r = forward_check("D3", 4, {"D3": 4}, step0, constraints)
assert r["D1"] == [1, 2, 3]           # 4 was never there
assert r["D2"] == [1, 2, 3, 5, 6, 7]
assert r["D4"] == [1, 2, 3, 5, 6, 7]
assert r["D5"] == [5, 6]

# Test 5: chained -- feed Test 1's result into the next call
r1 = forward_check("D1", 3, {"D1": 3}, step0, constraints)
r2 = forward_check("D4", 2, {"D1": 3, "D4": 2}, r1, constraints)
assert r2["D2"] == [1]
assert r2["D3"] == [1, 4]
assert r2["D5"] == [4, 5, 6]

# Test 6: emptiness from exclusivity alone, no precedence involved
tiny = {"A": [1], "B": [1]}
assert forward_check("A", 1, {"A": 1}, tiny, {"latest": {}, "precedences": []}) is None

print("all forward_check tests passed")

all forward_check tests passed


## Select the next task: MRV and degree

The Minimum Remaining Values (MRV) heuristic chooses an unassigned task with the fewest legal slots, which exposes failures early. If tasks tie, the degree heuristic chooses the one connected to the most still-unassigned tasks by precedence constraints. The task label provides a deterministic final tie-break.

In [26]:
def select_unassigned_variable(assignment, domains, constraints):
    unassigned = [task for task in domains if task not in assignment]
    if not unassigned:
        return None

    smallest = min(len(domains[task]) for task in unassigned)
    tied = [task for task in unassigned if len(domains[task]) == smallest]
    return min(tied, key=lambda task: (-degree(task, assignment, constraints), int(task[1:])))



def degree(task, assignment, constraints):
    count = 0
    for before, after in constraints["precedences"]:
        if before == task and after not in assignment:
            count += 1
        elif after == task and before not in assignment:
            count += 1
    return count

            

            
    




In [27]:
print(degree)                                          # should say: <function degree ...>
print(degree("D1", {}, constraints))                   # should print 2
print(select_unassigned_variable.__code__.co_varnames) # should include 'tied'

<function degree at 0x00000138280EC4A0>
2
('assignment', 'domains', 'constraints', 'unassigned', 'tied')


## Order candidate slots: LCV

The optional Least Constraining Value heuristic tries slots that remove the fewest choices from other tasks first. `forward_check` predicts the pruning for each candidate; choices causing an empty domain are ranked last. The numeric slot breaks ties consistently.

In [28]:
def order_domain_values(variable, assignment, domains, constraints):
    def cost(value):
        result = forward_check(variable, value, assignment, domains, constraints)
        if result is None:
            return float("inf")
        removed = sum(
            len(domains[task]) - len(result[task])
            for task in domains
            if task != variable
        )
        return removed
    return sorted(domains[variable], key=lambda v: (cost(v), v))

## Backtracking search

The recursive solver selects a task, tries candidate slots in LCV order, checks consistency, then forward-checks the remaining domains. If a branch fails, it removes that task's tentative assignment and tries the next value. Since each forward-check call returns copied domains, earlier domain lists are restored naturally when recursion returns.

In [29]:
def backtracking_search(variables, domains, constraints):
    def backtrack(assignment, domains):
        if len(assignment) == len(variables):
            return assignment
        var = select_unassigned_variable(assignment, domains, constraints)
        for value in order_domain_values(var, assignment, domains, constraints):
            print("  " * len(assignment), var, "=", value)
            if is_consistent(var, value, assignment, constraints):
                assignment[var] = value
                new_domains = forward_check(var, value, assignment, domains, constraints)
                if new_domains is not None:
                    result = backtrack(assignment, new_domains)
                    if result is not None:
                        return result
                del assignment[var]            # undo
        return None
    return backtrack({}, {v: list(domains[v]) for v in variables})

## Run and interpret the result

The main cell reloads the environment and schedule file, applies the latest-slot limits, assembles the constraints, and calls the solver. A returned dictionary maps every task to its slot; `None` means no feasible schedule was found. The printed schedule lists tasks in time order. With the current inputs, the expected result is `T1: D2`, `T2: D4`, `T3: D1`, `T4: D3`, and `T5: D5`; T6 and T7 are unused.

In [30]:
def main():
    environment_path = workspace_root / "notebook_cells" / "Environment.txt"
    schedule_path = workspace_root / "notebook_cells" / "schedule.txt"

    grid = load_environment(environment_path)
    variables = [f"D{i + 1}" for i in range(len(grid.dirty_locations))]
    domains, precedences, latest = load_schedule(variables, schedule_path)
    domains = apply_latest_slot(domains, latest)
    constraints = {"precedences": precedences, "latest": latest}

    schedule = backtracking_search(variables, domains, constraints)
    if schedule is None:
        print("No feasible schedule exists for schedule.txt.")
        return None

    print("Generated schedule:")
    for task, slot in sorted(schedule.items(), key=lambda item: item[1]):
        print(f"T{slot}: {task}")
    return schedule


if __name__ == "__main__":
    schedule = main()

 D1 = 2
   D4 = 1
 D1 = 3
   D4 = 2
     D2 = 1
       D3 = 4
         D5 = 5
Generated schedule:
T1: D2
T2: D4
T3: D1
T4: D3
T5: D5
